# 00 Run pipeline

Rebuilds every layer from `samples.tpch` and checks the result.

| Step | What it does | Output |
|---|---|---|
| Bronze | Copies the 8 source tables as is, plus `_ingested_at` | `workspace.bronze.*` |
| Silver | Typed 3NF tables with enforced quality, rejected rows go to quarantine | `workspace.silver.*` _(todo)_ |
| Gold | Star schema and headline metrics for the CTO questions | `workspace.gold.*` _(todo)_ |
| Validation | Runs all data-quality rules and shows pass/fail | results table below |

Names (catalog, source, as-of date) come from `src/tpch/config.py` and can be overridden with the `TPCH_CATALOG`, `TPCH_SOURCE` and `TPCH_AS_OF_DATE` environment variables.

In [0]:
%load_ext autoreload
%autoreload 2

## Run the pipeline

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))


from tpch import bronze, silver, gold, validation
bronze.run(spark)
silver.run(spark)
gold.run(spark)

Executing revenue drop anomaly check across Gold layer...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


INFO: No revenue anomalies detected. All metrics are operating within expected thresholds.


False

In [0]:
%sql
SHOW SCHEMAS in workspace;

databaseName
bronze
default
gold
information_schema
lakehouse_markus_pn
silver


## Validate
Every rule returns the number of violating rows. 0 means pass. See `src/tpch/validation.py`.

In [0]:
display(validation.run(spark))

layer,rule,violations,passed,checked_at
bronze,region: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,nation: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,supplier: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,customer: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,part: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,partsupp: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,orders: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,lineitem: row count equals source,0,true,2026-10-04T12:22:45.438Z
bronze,orders: customer exists,0,true,2026-10-04T12:22:45.438Z
bronze,lineitem: order exists,0,true,2026-10-04T12:22:45.438Z


## Anomaly Detection & Alerts
Monitors operational and financial metrics for unexpected drops or anomalies, specifically checking week-over-week (WoW) net revenue decreases exceeding 20 percent. See `src/tpch/alerts.py`.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from tpch.config import CATALOG

def run_alert(spark_session):
    print("Executing revenue drop anomaly check across Gold layer...")
    
    revenue_df = spark_session.sql(f"""
        SELECT 
            week_start, 
            SUM(net_revenue) AS weekly_revenue
        FROM {CATALOG}.gold.revenue_metrics
        GROUP BY week_start
        ORDER BY week_start
    """)

    window_spec = Window.orderBy("week_start")
    
    wow_analysis_df = revenue_df \
        .withColumn("previous_week_revenue", F.lag("weekly_revenue", 1).over(window_spec)) \
        .withColumn(
            "wow_change_percentage", 
            (F.col("weekly_revenue") - F.col("previous_week_revenue")) / F.col("previous_week_revenue")
        )

    anomaly_alerts_df = wow_analysis_df.filter(F.col("wow_change_percentage") < -0.20)
    anomaly_count = anomaly_alerts_df.count()
    
    if anomaly_count > 0:
        print(f"WARNING: Revenue anomaly detected. Found {anomaly_count} week(s) with WoW drop exceeding 20 percent.")
        display(anomaly_alerts_df)
        return True
    else:
        print("INFO: No revenue anomalies detected. All metrics are operating within expected thresholds.")
        return False

run_alert(spark)

Executing revenue drop anomaly check across Gold layer...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


INFO: No revenue anomalies detected. All metrics are operating within expected thresholds.


False

## Master Pipeline Orchestrator
Sequentially executes the end-to-end Lakehouse workflow (Bronze ingestion, Silver 3NF transformation, Gold data mart aggregation, data quality validation, and anomaly monitoring) with built-in error handling. See `src/tpch/orchestrator.py`.

In [0]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

from tpch import orchestrator

orchestrator.run_full_pipeline(spark)

STARTING TPC-H LAKEHOUSE PIPELINE EXECUTION
Target Catalog: workspace

[Step 1/5] Initializing Lakehouse catalog schemas...
SUCCESS: Schemas verified and ready.

[Step 2/5] Executing Bronze layer ingestion...
SUCCESS: Bronze layer ingestion completed.

[Step 3/5] Executing Silver layer 3NF transformation...
SUCCESS: Silver layer transformation completed.

[Step 4/5] Executing Gold layer data mart aggregation...
Executing revenue drop anomaly check across Gold layer...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


INFO: No revenue anomalies detected. All metrics are operating within expected thresholds.
SUCCESS: Gold layer aggregation completed.

[Step 5/5] Executing Data Quality validation and monitoring...


DataFrame[layer: string, rule: string, violations: bigint, passed: boolean, checked_at: timestamp]

SUCCESS: All Data Quality validation rules passed successfully.
Executing revenue drop monitoring query across Gold layer metrics...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


INFO: No revenue drop anomalies detected. All weekly metrics are operating within expected thresholds.

PIPELINE EXECUTION COMPLETED SUCCESSFULLY
